In [32]:
import os
import sys
import logging

logging.basicConfig(stream=sys.stdout, level=logging.DEBUG)

os.environ["CUDA_VISIBLE_DEVICES"] = "7"
#os.environ["TRITON_DEBUG"] = "1"
#os.environ["FLASH_ATTENTION_TRITON_AMD_DEBUG"] = "1"
os.environ["FLASH_ATTENTION_TRITON_AMD_AUTOTUNE"] = os.environ["TRITON_PRINT_AUTOTUNING "] = "1"

import math
import time
from tqdm import tqdm
import torch

from transformers.modeling_flash_attention_utils import _flash_attention_forward, attention_vanilla_forward_pytorch_ref_impl
from transformers.triton_flash_attention_fp8 import attention_backward_triton_impl
from transformers.triton_hadamard_transform import hadamard_transform

from torchtitan.logging import logger

model_type = "llama3-8b"
log_step = 10
max_step = 10
exclude_input_cvt = False
use_fp8 = True
eval_bwd = True
with_outliers = True
use_current_scaling = False
use_hadamard = False
uniform_dist = False

torch_dtype = torch.bfloat16
e4m3_dtype = torch.float8_e4m3fnuz
e5m2_dtype = torch.float8_e5m2fnuz
device = torch.device("cuda")

def rmse(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return torch.sqrt(torch.mean((x - y)**2))

def prepare_data(model_type: str, device: torch.device, with_outliers: bool = False):
    torch.manual_seed(0)

    configs = {
        "opt-125m": {
            "seqlen": 2048,
            "n_head": 12,
            "n_head_kv": 12,
            "head_dim": 64,
        },
        "llama2-7b": {
            "seqlen": 4096,
            "n_head": 32,
            "n_head_kv": 32,
            "head_dim": 128,
        },
        "llama2-70b": {
            "seqlen": 4096,
            "n_head": 64,
            "n_head_kv": 64,
            "head_dim": 128,
        },
        "llama3-8b": {
            "seqlen": 8192,
            "n_head": 32,
            "n_head_kv": 8,
            "head_dim": 128,
        },
        "llama3-70b": {
            "seqlen": 8192,
            "n_head": 64,
            "n_head_kv": 8,
            "head_dim": 128,
        },
    }

    c=configs[model_type]
    seqlen = c["seqlen"]
    n_head = c["n_head"]
    n_head_kv = c["n_head_kv"]
    head_dim = c["head_dim"]
    batch_size = 1
    query_states = torch.randn((batch_size, seqlen, n_head, head_dim),
                               dtype=torch_dtype,
                               device=device)
    key_states = torch.randn((batch_size, seqlen, n_head_kv, head_dim),
                             dtype=torch_dtype,
                             device=device)
    value_states = torch.randn((batch_size, seqlen, n_head_kv, head_dim),
                               dtype=torch_dtype,
                               device=device)
    output_states = torch.randn((batch_size, seqlen, n_head, head_dim),
                                dtype=torch_dtype,
                                device=device)
    softmax_lse = torch.randn((batch_size, n_head, seqlen),
                              device=device,
                              dtype=torch.float32)
    sm_scale = 1 / math.sqrt(head_dim)

    # outliers
    if with_outliers:
        if not uniform_dist:
            outlier_idx = torch.randint(0, head_dim, (1, )).to(device=device)
            sequence_p = torch.ones((1, seqlen, 1, 1), device=device) * 0.001 * batch_size * n_head * head_dim
            p_mask = torch.bernoulli(sequence_p)
            outlier_dist = 100 * torch.randn(size=(batch_size, seqlen, 1, 1)).to(
                dtype=torch_dtype, device=device)
            query_states[:, :, :, outlier_idx] += outlier_dist * p_mask
            outlier_dist = 100 * torch.randn(size=(batch_size, seqlen, 1, 1)).to(
                dtype=torch_dtype, device=device)
            key_states[:, :, :, outlier_idx] += outlier_dist * p_mask
            outlier_dist = 100 * torch.randn(size=(batch_size, seqlen, 1, 1)).to(
                dtype=torch_dtype, device=device)
            value_states[:, :, :, outlier_idx] += outlier_dist * p_mask
        else:
            r = 1 / (batch_size * n_head * head_dim)

            p_mask = torch.bernoulli(
                torch.ones(batch_size,
                        seqlen,
                        n_head,
                        head_dim,
                        dtype=torch_dtype,
                        device=device) * 0.001 * r)
            outlier_dist = 100 * torch.randn(size=(batch_size, seqlen, n_head,
                                                head_dim)).to(dtype=torch_dtype,
                                                                device=device)
            query_states += outlier_dist * p_mask

            r = 1 / (batch_size * n_head_kv * head_dim)
            p_mask = torch.bernoulli(
                torch.ones(batch_size,
                        seqlen,
                        n_head_kv,
                        head_dim,
                        dtype=torch_dtype,
                        device=device) * 0.001 * r)
            outlier_dist = 100 * torch.randn(size=(batch_size, seqlen, n_head_kv,
                                                head_dim)).to(dtype=torch_dtype,
                                                                device=device)
            key_states += outlier_dist * p_mask

            p_mask = torch.bernoulli(
                torch.ones(batch_size,
                        seqlen,
                        n_head_kv,
                        head_dim,
                        dtype=torch_dtype,
                        device=device) * 0.001* r)
            outlier_dist = 100 * torch.randn(size=(batch_size, seqlen, n_head_kv,
                                                head_dim)).to(dtype=torch_dtype,
                                                                device=device)
            value_states += outlier_dist * p_mask


    return query_states, key_states, value_states, output_states, softmax_lse, sm_scale


def check_and_convert_fp8(t, descale):
    finfo = torch.finfo(e4m3_dtype)
    return ((t * descale).clamp(min=finfo.min, max=finfo.max).to(e4m3_dtype)
            if t.dtype != e4m3_dtype else t)

def backward(do, q, k, v, o, softmax_lse, fp8_scales):
    q_scale, k_scale, v_scale, p_scale, _ = fp8_scales
    use_fp8 = q_scale is not None
    seqlen = q.shape[1]
    if use_fp8:
        float8_fw = e4m3_dtype
        float8_bw = e5m2_dtype

        def check_and_convert(t, scale):
            finfo = torch.finfo(float8_bw)
            return ((t * scale).clamp(min=finfo.min, max=finfo.max).to(
                dtype=float8_bw)
                    if t.dtype not in [float8_bw, float8_fw] else t)

        do_scale = torch.finfo(float8_bw).max / do.max()

        # TODO: determine o_scale outside this function
        o_scale = torch.finfo(float8_bw).max / o.max()

        do = check_and_convert(do, do_scale)
        q = check_and_convert(q, q_scale)
        k = check_and_convert(k, k_scale)
        v = check_and_convert(v, v_scale)
        o = check_and_convert(o, o_scale)
    else:
        do_scale = torch.tensor([1.], device=q.device)
        q_scale = torch.tensor([1.], device=q.device)
        k_scale = torch.tensor([1.], device=q.device)
        v_scale = torch.tensor([1.], device=q.device)
        o_scale = torch.tensor([1.], device=q.device)
        p_scale = torch.tensor([1.], device=q.device)

    din = attention_backward_triton_impl(
        do,
        q,
        k,
        v,
        o,
        q_scale,
        k_scale,
        v_scale,
        p_scale,
        o_scale,
        do_scale,
        softmax_lse,
        None,
        None,
        None,
        sm_scale,
        None,
        True,
        "bshd",
        0,
        0,
        seqlen,
        seqlen,
        True,
        use_fp8,
        sequence_parallel=True,
    )
    return din

query_states, key_states, value_states, output_states, softmax_lse, sm_scale = prepare_data(model_type, device, with_outliers)
query_states_hp = query_states.clone().detach()
key_states_hp = key_states.clone().detach()
value_states_hp = value_states.clone().detach()

if eval_bwd:
    loss_fn = torch.nn.MSELoss()
    query_states = torch.nn.Parameter(query_states, requires_grad=True)
    key_states = torch.nn.Parameter(key_states, requires_grad=True)
    value_states = torch.nn.Parameter(value_states, requires_grad=True)
    query_states_hp = torch.nn.Parameter(query_states_hp, requires_grad=True)
    key_states_hp = torch.nn.Parameter(key_states_hp, requires_grad=True)
    value_states_hp = torch.nn.Parameter(value_states_hp, requires_grad=True)

if use_hadamard:
    query_states_r = hadamard_transform(query_states)
    key_states_r = hadamard_transform(key_states)
else:
    query_states_r = query_states
    key_states_r = key_states

with torch.no_grad():
    range_q = torch.max(torch.abs(query_states_r))
    range_k = torch.max(torch.abs(key_states_r))
    range_v = torch.max(torch.abs(value_states))
    range_o = torch.max(torch.abs(output_states))
    range_p = 1
    num_tokens = query_states.shape[0] * query_states.shape[1]
    query_length = query_states.shape[1]

    descale_q = descale_k = descale_v = None
    if use_fp8:
        dtype_max = torch.finfo(e4m3_dtype).max

        descale_q = dtype_max / range_q
        descale_k = dtype_max / range_k
        descale_v = dtype_max / range_v
        descale_p = torch.scalar_tensor(240.0, device=query_states.device)
        descale_o = torch.scalar_tensor(1.0, device=query_states.device)
        fp8_scales = (descale_q, descale_k, descale_v, descale_p, descale_o)
        if exclude_input_cvt and not use_current_scaling:
            query_states_r = check_and_convert_fp8(query_states_r, descale_q)
            key_states_r = check_and_convert_fp8(key_states_r, descale_k)
            value_states = check_and_convert_fp8(value_states, descale_v)
    else:
        fp8_scales = (None, None, None, None, None)
    print(f"descale_q={descale_q}, descale_k={descale_k}, descale_v={descale_v}")
fa_output = _flash_attention_forward(
    query_states_r,
    key_states_r,
    value_states,
    None,
    query_length,
    True,
    0.0,
    softmax_scale=sm_scale,
    descale_q=descale_q,
    descale_k=descale_k,
    descale_v=descale_v,
    use_current_scaling=use_current_scaling,
)
print(fa_output.dtype)
fa_output_hp = attention_vanilla_forward_pytorch_ref_impl(
    query_states_hp, key_states_hp, value_states_hp, sm_scale, True, "bshd",
    False)[0]

if eval_bwd:
    if False:
        backward(output_states, query_states, key_states, value_states,
                 fa_output, softmax_lse, fp8_scales)
    else:
        loss = fa_output.mean()  #loss_fn(fa_output, output_states)
        loss.backward()

        loss_hp = fa_output_hp.mean() #loss_fn(fa_output_hp, output_states)
        loss_hp.backward()
e = rmse(fa_output, fa_output_hp)
print(f"fwd rmse(o)={e}, relative={e / torch.mean(fa_output_hp)}")
if eval_bwd:
    e = rmse(query_states.grad, query_states_hp.grad)
    print(f"bwd rmse(dq)={e}, relative={e / torch.mean(query_states_hp)}")
    e = rmse(key_states.grad, key_states_hp.grad)
    print(f"bwd rmse(dk)={e}, relative={e / torch.mean(key_states_hp)}")
    e = rmse(value_states.grad, value_states_hp.grad)
    print(f"bwd rmse(dv)={e}, relative={e / torch.mean(value_states_hp)}")

descale_q=0.58203125, descale_k=0.55078125, descale_v=0.6171875
torch.bfloat16
fwd rmse(o)=1.703125, relative=3.296875
bwd rmse(dq)=0.00107574462890625, relative=-0.66015625
bwd rmse(dk)=0.025634765625, relative=-15.875
bwd rmse(dv)=0.00058746337890625, relative=-0.0771484375


In [24]:
torch.mean(fa_output_hp)

tensor(-0.0011, device='cuda:0', dtype=torch.bfloat16, grad_fn=<MeanBackward0>)

In [19]:
sequence_p = torch.ones((1, 8192, 1, 1), device=device) * 0.001
p_mask = torch.bernoulli(sequence_p)
torch.sum(p_mask)

tensor(5., device='cuda:0')

In [2]:
print(query_states.grad.shape)


torch.Size([1, 8192, 32, 128])


In [ ]:

torch.cuda.synchronize()
time_last_log = time.perf_counter()
tps_list = []

for i in tqdm(range(max_step)):
    fa_output = _flash_attention_forward(
        query_states,
        key_states,
        value_states,
        None,
        query_length,
        True,
        0.0,
        softmax_scale=sm_scale,
        descale_q=descale_q,
        descale_k=descale_k,
        descale_v=descale_v,
        use_current_scaling=use_current_scaling,
    )
    if eval_bwd:
        #if use_fp8 and exclude_input_cvt:
        if True:
            backward(output_states, query_states, key_states, value_states,
                     fa_output, softmax_lse, fp8_scales)
        else:
            loss = loss_fn(fa_output, output_states)
            loss.backward()
    torch.cuda.synchronize()

    time_delta = time.perf_counter() - time_last_log
    # tokens per second, abbreviated as tps
    tps = num_tokens / time_delta
    tps_list.append(tps)
    # if i % log_step == 0:
    #     logger.info(f"tps: {tps}")
    time_last_log = time.perf_counter()

logger.info(f"mean tps: {torch.tensor(tps_list).mean()}")


  0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
from transformers.triton_flash_attention_fp8 import attn_fwd, _bwd_kernel

cached_kernel = (list(_bwd_kernel.fn.cache.values())[0])
kernel = (list(cached_kernel.values())[0])
# kernel = list(attn_fwd.fn.device_caches[0][0].values())[0]
# #print(print(_bwd_kernel.fn.device_caches[0]))
# # print(print(
# #     list(_bwd_kernel.fn.device_caches[0][0].values())[0].asm['amdgcn']))
# # # _bwd_kernel.fn.compile()
# print(kernel.asm.keys())

In [ ]:
print(kernel.asm['amdgcn'])

	.text
	.amdgcn_target "amdgcn-amd-amdhsa--gfx942"
	.amdhsa_code_object_version 4
	.globl	_bwd_kernel                     ; -- Begin function _bwd_kernel
	.p2align	8
	.type	_bwd_kernel,@function
_bwd_kernel:                            ; @_bwd_kernel
.Lfunc_begin0:
	.cfi_sections .debug_frame
	.cfi_startproc
	s_trap 2 ; Kernarg preload header. Trap with incompatible firmware that doesn't support preloading kernel arguments.
	.fill 63, 4, 0xbf800000 ; s_nop 0
; %bb.0:
	.file	1 "/workspace/transformers/src/transformers" "triton_flash_attention_fp8.py"
	.loc	1 1563 26 prologue_end          ; triton_flash_attention_fp8.py:1563:26
	s_load_dwordx2 s[16:17], s[0:1], 0x40
	v_mov_b32_e32 v97, 0
	global_load_ushort v85, v97, s[8:9]
	.loc	1 1564 26                       ; triton_flash_attention_fp8.py:1564:26
	global_load_ushort v78, v97, s[10:11]
	.loc	1 1565 26                       ; triton_flash_attention_fp8.py:1565:26
	global_load_ushort v61, v97, s[12:13]
	.loc	1 1568 27                    